To use the radon operator the astra toolbox needs to be installed this is possible via 
```
conda install astra-toolbox -c astra-toolbox
```
for other methods see https://astra-toolbox.com/docs/install.html

also `scikit-image` needs to be installed


# Computer Tomography (CT) and Radon transform
blabla

In [ ]:
from examples.radon_transform.radon_astra import RadonAstra2D,RadonMatrixAstra2D
import numpy as np
import matplotlib.pyplot as plt
from skimage.data import shepp_logan_phantom
from skimage.transform import rescale



In [ ]:
def Get2DBall(x,y,radius,x_res,y_res=None,inner_radius = 0):
    """creates a imige 2D of a ball around (x,y) with the radius,
      wich is 1 on the ball and 0 everywhere else.

    Args:
        x (float): x position of ball 
        y (float): y position of ball
        radius (float): radius in pixel count
        x_res (int): number of pixels in x diretion
        y_res (int, optional): number of pixels in y direction. Defaults to x_res.

    Returns:
        nparray: _description_
    """    
    if y_res is None:
        y_res = x_res

    ball = np.zeros((y_res,x_res))
    for i in range(y_res):
        for j in range(x_res):
            if inner_radius**2<=(i-x)**2+(j-y)**2<= radius**2:
                ball[i,j] = 1
    return ball



In [ ]:
#creating the phantom
num_pix = 128
phantome = shepp_logan_phantom()
phantome = rescale(phantome, scale=num_pix/400, mode='reflect', channel_axis=None)
ball_phant = Get2DBall(22.5,30.5,4,num_pix,inner_radius=3) + Get2DBall(22.5,30,8,num_pix,inner_radius=7)+Get2DBall(40.5,20,8,num_pix,inner_radius=0) - Get2DBall(43.5,23,2,num_pix,inner_radius=0)*0.2
ball_phant = ball_phant * Get2DBall(num_pix/2-0.5,num_pix/2-0.5,num_pix/2,num_pix)

In [ ]:
angles = np.linspace(0,np.pi,num_pix,endpoint=False)

In [ ]:
#Create a linear operator based on a sparse matrix created using the astra toolbox
op = RadonMatrixAstra2D(num_pix=num_pix,num_det=int(num_pix*np.sqrt(2)),
                  angles=angles,
                  geom_type="parallel",beam_type="strip",
                  source_to_origin=128,origin_to_detector=128,dx =1)



In [ ]:
#create a linear operator based on opeartors of the astra tolbox (this failes the adjoint test)
op_ast = RadonAstra2D(num_pix=num_pix,num_det=int(num_pix*np.sqrt(2)),
                  angles=angles,
                  geom_type="parallel",beam_type="strip",
                  source_to_origin=128,origin_to_detector=128,dx =1)

In [ ]:
sinogramm = op(phantome)
plt.figure()
plt.imshow(sinogramm)

In [ ]:
from regpy.solvers import RegularizationSetting
import regpy.stoprules as rules
from regpy.hilbert import L2
# from regpy.functionals import L1
from regpy.solvers.linear.tikhonov import TikhonovCG

In [ ]:
# import logging
# logging.basicConfig(
#     level=logging.INFO,
#     format='%(asctime)s %(levelname)s %(name)-40s :: %(message)s'
# )

In [ ]:
stoprule = (
    rules.CountIterations(max_iterations=100)
)
setting = RegularizationSetting(op,L2,L2)
solver = TikhonovCG(setting,sinogramm,0)
reco, reco_data = solver.run(stoprule)

In [ ]:
plt.figure()
plt.imshow(reco,"gray_r")
plt.colorbar()

In [ ]:
stoprule = (
    rules.CountIterations(max_iterations=100)
)
setting = RegularizationSetting(op_ast,L2,L2)
solver = TikhonovCG(setting,sinogramm,0)
reco, reco_data = solver.run(stoprule)

In [ ]:
plt.figure()
plt.imshow(reco)
plt.colorbar()

In [ ]:
from regpy.solvers.linear.primal_dual import PDHG,DouglasRachford
from regpy.functionals import L1Generic,L1MeasureSpace,Functional
from regpy.functionals import HilbertNorm, TVUniformGridFcts

In [ ]:
class NonNeg(Functional):
    def __init__(self, domain, h_domain=None, linear=False):
        super().__init__(domain, h_domain, linear)
    
    def proximal(self, x, tau, recursion_safeguard=False):
        return np.maximum(np.sign(x)*np.maximum(np.abs(x)-tau,0),0)
    


In [ ]:
from regpy.operators import SciPyLinearOperator
from scipy.linalg.interpolative import estimate_spectral_norm

In [ ]:
op_norm = estimate_spectral_norm(SciPyLinearOperator(op))
op_norm

In [ ]:
data = sinogramm
init = op.domain.ones()
setting = RegularizationSetting(op,
                                penalty=NonNeg,
                                data_fid=HilbertNorm(h_space=L2) * (op - data)
                                )
stoprule = (
    rules.CountIterations(max_iterations=100)+
    rules.Discrepancy(setting.h_codomain.norm, data,0.001)
            )
solver = DouglasRachford(setting,init,tau = 0.99,regpar=1)
reco, reco_data = solver.run(stoprule)

In [ ]:
plt.figure()
plt.imshow(reco,cmap="gray_r")
plt.colorbar()

In [ ]:
init = op.domain.ones()
setting = RegularizationSetting(op_ast,
                                penalty=NonNeg,
                                data_fid=HilbertNorm(h_space=L2) * (op_ast - data)
                                )
stoprule = (
    rules.CountIterations(max_iterations=100)+
    rules.Discrepancy(setting.h_codomain.norm, data,0.01)
            )
solver = DouglasRachford(setting,init,tau = 0.01,regpar=0.01)
reco, reco_data = solver.run(stoprule)

In [ ]:
plt.figure()
plt.imshow(reco)
plt.colorbar()

In [ ]:
setting.penalty.proximal?

In [ ]:
from regpy.functionals import L1Generic 


In [ ]:
from skimage.transform import radon, iradon

In [ ]:
data = radon(image,angles*180/np.pi)
noise= np.random.rand(*data.shape)

In [ ]:
plt.imshow(iradon(data+noise*10,angles*180/np.pi,filter_name="cosine"))

In [ ]:
plt.imshow(iradon(data+noise*10,angles*180/np.pi,filter_name="ramp"))

In [ ]:
rec = iradon(data+noise*10,angles*180/np.pi,filter_name="hann")
plt.imshow(rec)

In [ ]:
res = num_pix 
ball = Get2DBall(res/2,res/2,res/20,res)
plt.imshow(ball)

In [ ]:
res = num_pix 
ball = Get2DBall(res/2,res/2,res/8,res)
from numpy.fft import fft,ifft,fft2,ifft2,fftshift,ifftshift
f_img = fft2(rec)
f_img = fftshift(f_img)
f_img = ball * f_img
f_img = ifftshift(f_img)
f_img = ifft2(f_img)

plt.imshow(np.abs(f_img))
plt.colorbar()

In [ ]:
class ProjDeltaBall(): 
    def __init__(self,mid_point,delta) -> None:
        self.g = mid_point
        self.norm_g = np.linalg.norm(self.g.flatten())
        self.delta = delta

    def proj(self,y,mid_point,delta = None):
        delta = delta or self.delta
        delta = delta*self.norm_g
        h = y-mid_point
        h_length = np.linalg.norm(h.flatten())
        if h_length <= delta:
            return y
        else:
            return mid_point + delta/h_length*h
        
    def __call__(self,x):
        return self.proj(x,self.g,self.delta)

In [ ]:
data_delta = data + 5*noise
delta = np.linalg.norm((data_delta-data).flatten())/np.linalg.norm((data_delta).flatten())
proj = ProjDeltaBall(data_delta,delta)

In [ ]:
def low_pass_filter(img,alpha):
    assert img.shape[0] == img.shape[1]
    res = img.shape[0] 
    ball = Get2DBall(res/2,res/2,res/alpha,res)
    f_img = fft2(img)
    f_img = fftshift(f_img)
    f_img = ball * f_img
    f_img = ifftshift(f_img)
    f_img = ifft2(f_img)
    return np.abs(f_img)

In [ ]:
f = (iradon(data_delta,angles*180/np.pi))*0
proj = ProjDeltaBall(data_delta,delta)
recs_list = []
loss = []
for i in range(100):
    y = radon(f,angles*180/np.pi)
    p = proj(y)
    f_full = iradon(p,angles*180/np.pi,filter_name="hann")
    f = low_pass_filter(f_full,10)
    f = np.fmax(f-0.1,0)
    recs_list.append(f_full)
    loss.append(np.linalg.norm((f-image).flatten()))

In [ ]:
plt.imshow(f_full)
plt.colorbar()

In [ ]:
plt.plot(loss)
plt.yscale("log")

In [ ]:
# np.save("view.npy",recs_list)

In [ ]:
delta